# R15-H157 - Corpus-class transfer: the single-corpus overfit test

**Hypothesis.** On a structurally different technical corpus (scientific-paper library),
the engine's lifecycle machinery *transfers* (curing gate fires and holds, FSM reaches
STABLE, no spurious recures) but the identity calibration does *not* transfer within
tolerance (ECE degrades > 2x). Calibration must be re-estimated per corpus.

**Prediction.** Lifecycle transfers; calibration breaks; the deterministic detectors
(name-identity, model-code) carry different false-merge surfaces per domain.

**Acceptance bar.** Both clauses measured on >= 20 documents of a second corpus.
REFUTED (pleasantly) if calibration holds within 2x ECE.

Corpus: first 30 PDFs (sorted filename) of `references/papers/`. Engine: local vLLM
(gpt-oss-120b) for extraction, Bedrock Titan embeddings UNCHANGED (the shipped isotonic
curve is calibrated on Titan cosine), `resolution.identity_stack: v2` with the shipped
`data/processed/identity-calibration-v2.json`. Graph on the default scratch Neo4j.


## Imports

In [ ]:
from __future__ import annotations
import sys, json, math
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime, timezone

import numpy as np
from rich.console import Console
from rich.table import Table

sys.path.insert(0, "../src")
from knowledge_graph_foundry.settings import load_settings
from knowledge_graph_foundry.models import Entity, normalize_name
from knowledge_graph_foundry.resolution.similarity import cosine_similarity
from knowledge_graph_foundry.resolution.blocking import ann_candidates
from knowledge_graph_foundry.resolution.resolver import _fuzzy_candidates
from knowledge_graph_foundry.resolution.identity_stack import V2IdentityStack
from knowledge_graph_foundry.resolution.bayesian import evidence
from knowledge_graph_foundry.graph.aliases import code_tokens
from neo4j import GraphDatabase

console = Console()
rp = console.print

## Configuration

In [ ]:
CONFIG_PATH = Path("../config-h157-papers.yml")
ARTIFACT_PATH = Path("../data/processed/identity-calibration-v2.json")
EVENTS_PATH = Path("../logs/h157-events.jsonl")
REF_H129_REPORT = Path("../reports/calibration-h129-transfer-20260707-100316.json")
DOC_LIST_PATH = Path("/tmp/h157_list.txt")  # frozen 30-doc list

settings = load_settings(CONFIG_PATH)
artifact = json.loads(ARTIFACT_PATH.read_text())
stack = V2IdentityStack(artifact, settings.resolution.nli_veto_threshold)

# Reference denominator: original-corpus ECE of the SHIPPED isotonic curve (H129),
# computed on the 297-pair adjudicated benchmark with the same 10-bin ECE estimator.
REF_ECE = json.loads(REF_H129_REPORT.read_text())["ece"]
ECE_BAR = 2.0 * REF_ECE  # transfer FAILS if new-corpus ECE exceeds this

doc_list = [l.strip() for l in DOC_LIST_PATH.read_text().splitlines() if l.strip()]

t = Table(title="H157 configuration", show_header=False)
t.add_row("neo4j (pinned)", settings.neo4j.uri)
t.add_row("extraction llm", f"{settings.llm.engine} / {settings.llm.model}")
t.add_row("embeddings", f"{settings.embeddings.provider} / {settings.embeddings.model}")
t.add_row("identity_stack", settings.resolution.identity_stack)
t.add_row("logistic threshold", str(artifact["logistic"]["threshold"]))
t.add_row("nli veto", str(settings.resolution.nli_veto_threshold))
t.add_row("docs (frozen)", str(len(doc_list)))
t.add_row("REF ECE (H129, orig corpus)", f"{REF_ECE:.4f}")
t.add_row("ECE fail bar (2x)", f"{ECE_BAR:.4f}")
console.print(t)

## Clause 1 - lifecycle transfer

Parse the ingest event log for the curing gate (Good-Turing missing-mass UCB
trajectory, cure document index, hold), the FSM state trajectory, drift alarms,
and the type inventory at cure vs end.

In [ ]:
def load_events(path):
    return [json.loads(l) for l in Path(path).read_text().splitlines() if l.strip()]

events = load_events(EVENTS_PATH)

MMZ = settings.curing.missing_mass_z          # 1.64
MM_THR = settings.curing.missing_mass_threshold  # 0.05

def ucb(rec):
    tot = rec.get("total_occurrences"); n1 = rec.get("singletons")
    if not tot or n1 is None:
        return None
    n1 = float(n1)
    return n1 / tot + MMZ * ((n1 + 1.0) ** 0.5) / tot

# Run-aware segmentation: a killed-before-save ingest attempt can leave a
# duplicate doc-0 trace (extra CURING transition + re-emerged types) ahead of
# the surviving run. The surviving run's fluid phase begins at the LAST
# INITIALIZING->CURING transition (start_curing only fires once per run, and a
# resumed run already in CURING never re-emits it). Take events from there.
cure_starts = [k for k,e in enumerate(events)
               if e["event"] == "fsm.transition" and e["state"] == "CURING"]
run_start = cure_starts[-1] if cure_starts else 0
# include the CURING doc's type emergences that immediately precede it in-doc
run_events = events[run_start:]
# document.completed / drift are graph-level and only ever fire in the live run
docs_completed = [e for e in events if e["event"] == "document.completed"]

curing_metrics = [e for e in run_events if e["event"] == "curing.metrics"]
fsm_trans = [e for e in run_events if e["event"] == "fsm.transition"]
type_confirmed = [e["type"] for e in run_events if e["event"] == "ontology.type_confirmed"]
drift_events = [e for e in run_events if e["event"].startswith("drift.")]
evolved = [e for e in run_events if e["event"] == "ontology.evolved"]

# global INITIALIZING prefix for display (init happened once, pre-run)
traj = ["INITIALIZING"]
for e in fsm_trans:
    if not traj or traj[-1] != e["state"]:
        traj.append(e["state"])
reached_stable = "STABLE" in traj
recures = sum(1 for e in fsm_trans if e["state"] == "RECURING")

cure_doc_index = None
seen_metrics = 0
for e in run_events:
    if e["event"] == "curing.metrics":
        seen_metrics += 1
    if e["event"] == "fsm.transition" and e["state"] == "STABLE" and cure_doc_index is None:
        cure_doc_index = seen_metrics - 1

ucb_traj = [(e["doc_index"], e.get("unique_types"), e.get("singletons"),
             e.get("total_occurrences"), ucb(e),
             e.get("chao1_coverage"), e.get("js_divergence"),
             e.get("entropy_shannon_delta")) for e in curing_metrics]

t = Table(title="Curing gate trajectory (Good-Turing UCB per document)")
for c in ["doc","types","n1","N","mm_UCB","chao1_cov","jsd","dH"]:
    t.add_column(c)
for di,ut,n1,N,u,ch,jsd,dh in ucb_traj:
    t.add_row(str(di), str(ut), str(n1), str(N),
              f"{u:.4f}" if u is not None else "-",
              f"{ch:.3f}" if ch is not None else "-",
              f"{jsd:.4f}" if jsd==jsd else "nan",
              f"{dh:.4f}" if dh is not None else "-")
console.print(t)

rp(f"[bold]FSM trajectory[/bold]: {' -> '.join(traj)}")
rp(f"reached STABLE: [bold]{reached_stable}[/bold] | cure doc index: [bold]{cure_doc_index}[/bold] | spurious recures: [bold]{recures}[/bold]")
rp(f"drift events post-cure: {len(drift_events)} | ontology.evolved stages: {[e.get('stage') for e in evolved]}")
rp(f"docs completed: {len(docs_completed)}")

In [ ]:
pre, post = [], []
seen_stable = False
for e in run_events:
    if e["event"] == "fsm.transition" and e["state"] == "STABLE":
        seen_stable = True
    elif e["event"] == "ontology.type_emerged":
        (post if seen_stable else pre).append(e["type"])

rp(f"[bold]Types emerged pre-cure[/bold] ({len(set(pre))}): {sorted(set(pre))}")
rp(f"[bold]Types emerged post-cure[/bold] ({len(set(post))}): {sorted(set(post))}")
lifecycle = {
    "fsm_trajectory": traj,
    "reached_stable": reached_stable,
    "cure_doc_index": cure_doc_index,
    "spurious_recures": recures,
    "cure_holds": reached_stable and recures == 0,
    "docs_completed": len(docs_completed),
    "types_pre_cure": sorted(set(pre)),
    "types_post_cure": sorted(set(post)),
    "n_types_pre_cure": len(set(pre)),
    "n_types_post_cure_emergent": len(set(post) - set(pre)),
    "drift_events": [{k:v for k,v in e.items() if k!='ts'} for e in drift_events],
    "ontology_evolved_stages": [e.get("stage") for e in evolved],
    "ucb_trajectory": [
        {"doc_index": di, "unique_types": ut, "singletons": n1,
         "total_occurrences": N, "missing_mass_ucb": u,
         "chao1_coverage": ch, "js_divergence": (jsd if jsd==jsd else None),
         "entropy_shannon_delta": dh}
        for di,ut,n1,N,u,ch,jsd,dh in ucb_traj],
    "mm_threshold": MM_THR,
}
print(json.dumps({k:lifecycle[k] for k in ("reached_stable","cure_doc_index","cure_holds","spurious_recures","n_types_pre_cure","n_types_post_cure_emergent")}, indent=1))

## Clause 2 - calibration transfer

Export candidate entity pairs from the resolver's own candidate-generation path
(fuzzy-name sorted-neighbourhood UNION within-type ANN blocking over Titan
embeddings), compute the shipped isotonic-calibrated cosine, then blind-label a
stratified sample and measure ECE.

In [ ]:
driver = GraphDatabase.driver(settings.neo4j.uri, auth=(settings.neo4j.user, settings.neo4j.password))

def load_graph_entities(driver):
    q = '''MATCH (e:Entity)
           RETURN e.id AS id, e.name AS name, labels(e) AS labels,
                  e.description AS description, e.embedding AS embedding,
                  e.source_chunks AS source_chunks, e.source_documents AS source_documents,
                  properties(e) AS props'''
    ents = []
    with driver.session() as s:
        for r in s.run(q):
            labels = [l for l in (r["labels"] or []) if l != "Entity"]
            props = dict(r["props"] or {})
            for k in ("id","name","description","embedding","source_chunks","source_documents"):
                props.pop(k, None)
            props = {k:v for k,v in props.items() if k not in ("updated_at","created_at","version","valid_from","valid_to")}
            ents.append(Entity(
                id=r["id"], name=r["name"] or "", types=labels or ["Entity"],
                description=r["description"] or "",
                embedding=list(r["embedding"]) if r["embedding"] else None,
                source_chunks=list(r["source_chunks"] or []),
                source_documents=list(r["source_documents"] or []),
                properties=props,
            ))
    return ents

def load_chunk_texts(driver, chunk_ids):
    if not chunk_ids: return {}
    q = "MATCH (c:Chunk) WHERE c.id IN $ids RETURN c.id AS id, c.text AS text"
    out = {}
    with driver.session() as s:
        for r in s.run(q, ids=list(chunk_ids)):
            out[r["id"]] = r["text"] or ""
    return out

entities = load_graph_entities(driver)
with_emb = [e for e in entities if e.embedding]
rp(f"entities in graph: [bold]{len(entities)}[/bold] | with embedding: [bold]{len(with_emb)}[/bold]")
type_counts = Counter(t for e in entities for t in e.types)
rp(f"distinct types: {len(type_counts)} | top: {type_counts.most_common(12)}")

In [ ]:
cfg = settings.resolution
cands = set()
cands |= _fuzzy_candidates(with_emb, 0.82)
cands |= ann_candidates(with_emb, cfg.ann_top_k, cfg.ann_min_entities, cfg.synonym_cluster_threshold)
cands = sorted(cands)
rp(f"candidate pairs (fuzzy-name UNION within-type ANN): [bold]{len(cands)}[/bold]")

def cos(i,j):
    return cosine_similarity(with_emb[i].embedding, with_emb[j].embedding)

cand_rows = []
for i,j in cands:
    c = cos(i,j)
    cc = stack.calibrated_cosine(c)
    a,b = with_emb[i], with_emb[j]
    cand_rows.append({
        "i": i, "j": j, "a_id": a.id, "b_id": b.id,
        "a_name": a.name, "b_name": b.name,
        "a_types": a.types, "b_types": b.types,
        "cosine": c, "calibrated_cosine": cc,
        "name_id": int(normalize_name(a.name) == normalize_name(b.name)),
    })
cc_vals = np.array([r["calibrated_cosine"] for r in cand_rows]) if cand_rows else np.array([])
if len(cc_vals):
    rp(f"calibrated-cosine spread: min {cc_vals.min():.3f} / med {np.median(cc_vals):.3f} / max {cc_vals.max():.3f}")

In [ ]:
rng = np.random.default_rng(157)
TARGET = 60
def stratified_sample(rows, target):
    if len(rows) <= target:
        return list(range(len(rows)))
    ccs = np.array([r["calibrated_cosine"] for r in rows])
    edges = np.array([0.0,0.1,0.2,0.3,0.4,0.5,0.65,0.8,0.9,1.001])
    idx_by_bin = defaultdict(list)
    for k,c in enumerate(ccs):
        b = int(np.searchsorted(edges, c, side="right")-1)
        idx_by_bin[b].append(k)
    nbins = len([b for b in idx_by_bin if idx_by_bin[b]])
    per = max(1, target // max(1,nbins))
    chosen = []
    for b, ks in sorted(idx_by_bin.items()):
        take = min(len(ks), per)
        chosen += list(rng.choice(ks, size=take, replace=False))
    remaining = [k for k in range(len(rows)) if k not in set(chosen)]
    rng.shuffle(remaining)
    while len(chosen) < target and remaining:
        chosen.append(remaining.pop())
    return sorted(set(int(x) for x in chosen))[:target]

sample_idx = stratified_sample(cand_rows, TARGET)
rp(f"sampled [bold]{len(sample_idx)}[/bold] pairs for blind labeling")

all_chunk_ids = set()
for k in sample_idx:
    all_chunk_ids |= set(with_emb[cand_rows[k]["i"]].source_chunks[:2])
    all_chunk_ids |= set(with_emb[cand_rows[k]["j"]].source_chunks[:2])
chunk_texts = load_chunk_texts(driver, all_chunk_ids)

def snippet(e, n=240):
    for cid in e.source_chunks[:2]:
        tx = chunk_texts.get(cid, "")
        if tx: return tx[:n].replace(chr(10)," ")
    return ""

evidence_dump = []
for pos,k in enumerate(sample_idx):
    r = cand_rows[k]; a,b = with_emb[r["i"]], with_emb[r["j"]]
    evidence_dump.append({
        "pair": pos, "cand_k": k,
        "a_name": a.name, "a_types": a.types, "a_desc": (a.description or "")[:200],
        "a_docs": [d for d in a.source_documents][:3], "a_snip": snippet(a),
        "b_name": b.name, "b_types": b.types, "b_desc": (b.description or "")[:200],
        "b_docs": [d for d in b.source_documents][:3], "b_snip": snippet(b),
    })
Path("/tmp/h157_evidence.json").write_text(json.dumps(evidence_dump, indent=1))
rp(f"evidence written for {len(evidence_dump)} pairs (scores withheld)")

### Blind labels (frozen)

Each pair labeled SAME (1) / DIFFERENT (0) from names, types, descriptions and source
snippets ONLY, before any cosine or model score was revealed (H101 blind protocol).

In [ ]:
FROZEN_LABELS = __FROZEN_LABELS__
assert len(FROZEN_LABELS) == len(sample_idx), (len(FROZEN_LABELS), len(sample_idx))

### ECE of the shipped calibrated posterior

In [ ]:
def ece(probs, y, bins=10):
    probs = np.asarray(probs, float); y = np.asarray(y, float)
    edges = np.linspace(0,1,bins+1); e = 0.0
    for i in range(bins):
        m = (probs >= edges[i]) & (probs < edges[i+1] if i < bins-1 else probs <= edges[i+1])
        if m.sum()==0: continue
        e += (m.sum()/len(probs)) * abs(probs[m].mean() - y[m].mean())
    return float(e)

labeled = [(cand_rows[sample_idx[p]], FROZEN_LABELS[p]) for p in range(len(sample_idx))]
y = np.array([lab for _,lab in labeled])
cc_probs = np.array([r["calibrated_cosine"] for r,_ in labeled])

ece_calcos = ece(cc_probs, y)

pairs_ent = [(with_emb[r["i"]], with_emb[r["j"]]) for r,_ in labeled]
nli = stack.nli_contra_batch(pairs_ent)
logit_probs = []
for (r,_), contra in zip(labeled, nli):
    a,b = with_emb[r["i"]], with_emb[r["j"]]
    post = evidence(a,b,cfg).posterior
    s = stack.score(a,b,post,contra)
    logit_probs.append(s)
logit_probs = np.array(logit_probs)
ece_logistic = ece(logit_probs, y)

rp(f"labeled sample: n={len(y)} | SAME={int(y.sum())} DIFFERENT={int((1-y).sum())}")
rp(f"[bold]ECE calibrated-cosine (new corpus)[/bold]: {ece_calcos:.4f}  (H129 orig-corpus ref {REF_ECE:.4f})")
rp(f"[bold]ECE full logistic posterior (new corpus)[/bold]: {ece_logistic:.4f}")
ece_ratio = ece_calcos / REF_ECE
rp(f"ECE ratio (calibrated-cosine / ref): [bold]{ece_ratio:.2f}x[/bold]  | fail bar 2.00x")
calibration_transfers = ece_calcos <= ECE_BAR

## Deterministic detector census on the paper domain

`name_id` (normalized-name equality) and the `model_code` detector (mixed
alphanumeric tokens - `code_tokens`). Enumerate the model-code false-merge surface
on paper-domain identifiers (metric notation F1/P@k, model/version strings, arXiv-like
tokens).

In [ ]:
name_id_fires = [r for r in cand_rows if r["name_id"] == 1]
rp(f"name_id fires on {len(name_id_fires)} / {len(cand_rows)} candidate pairs")

by_code = defaultdict(list)
for e in entities:
    for c in code_tokens(e.name):
        by_code[c].append((e.id, e.name, tuple(e.types)))
code_clusters = {c: v for c,v in by_code.items() if len(v) > 1}
rp(f"model_code tokens producing multi-entity clusters: [bold]{len(code_clusters)}[/bold]")

false_surface = []
for c, members in sorted(code_clusters.items(), key=lambda kv:-len(kv[1])):
    norms = set(normalize_name(m[1]) for m in members)
    types = set(t for m in members for t in m[2])
    if len(norms) > 1:
        false_surface.append({"code": c, "n_members": len(members),
                              "distinct_names": sorted(norms)[:6],
                              "types": sorted(types)[:6]})
rp(f"model_code clusters spanning DISTINCT names (false-merge risk): [bold]{len(false_surface)}[/bold]")
t = Table(title="model_code false-merge surface (top 15 by size)")
for c in ["code","members","distinct names (sample)"]:
    t.add_column(c, overflow="fold")
for f in false_surface[:15]:
    t.add_row(f["code"], str(f["n_members"]), ", ".join(f["distinct_names"]))
console.print(t)

## Clause evaluation and verdict

In [ ]:
clause1_pass = lifecycle["reached_stable"] and lifecycle["cure_holds"]
if clause1_pass and not calibration_transfers:
    verdict = "CONFIRMED"
elif clause1_pass and calibration_transfers:
    verdict = "REFUTED"
else:
    verdict = "PARTIAL"

rp(f"[bold]Clause 1 (lifecycle transfers)[/bold]: {'PASS' if clause1_pass else 'FAIL'}  "
   f"(STABLE={lifecycle['reached_stable']}, cure_holds={lifecycle['cure_holds']}, recures={lifecycle['spurious_recures']})")
rp(f"[bold]Clause 2 (calibration breaks)[/bold]: {'BREAKS -> hypothesis supported' if not calibration_transfers else 'HOLDS -> hypothesis refuted'}  "
   f"(ECE {ece_calcos:.4f} vs bar {ECE_BAR:.4f}, ratio {ece_ratio:.2f}x)")
rp(f"[bold]VERDICT[/bold]: {verdict}")

In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
report = {
    "hypothesis": "R15-H157",
    "title": "Corpus-class transfer - single-corpus overfit test",
    "stamp": stamp,
    "corpus": {
        "name": "scientific-paper library (references/papers)",
        "n_documents_requested": len(doc_list),
        "n_documents_completed": lifecycle["docs_completed"],
        "document_list": doc_list,
    },
    "engine": {
        "neo4j": settings.neo4j.uri,
        "extraction_llm": f"{settings.llm.engine}/{settings.llm.model}",
        "embeddings": f"{settings.embeddings.provider}/{settings.embeddings.model}",
        "identity_stack": settings.resolution.identity_stack,
        "artifact": str(ARTIFACT_PATH),
    },
    "clause1_lifecycle": lifecycle,
    "clause2_calibration": {
        "ref_ece_original_corpus_h129": REF_ECE,
        "ece_bar_2x": ECE_BAR,
        "new_corpus_ece_calibrated_cosine": ece_calcos,
        "new_corpus_ece_logistic_posterior": ece_logistic,
        "ece_ratio_calibrated_cosine": ece_ratio,
        "calibration_transfers_within_2x": bool(calibration_transfers),
        "n_candidate_pairs": len(cand_rows),
        "n_labeled": int(len(y)),
        "n_same": int(y.sum()),
        "n_different": int((1-y).sum()),
        "labeled_pairs": [
            {"a_name": r["a_name"], "b_name": r["b_name"],
             "a_types": r["a_types"], "b_types": r["b_types"],
             "cosine": r["cosine"], "calibrated_cosine": r["calibrated_cosine"],
             "name_id": r["name_id"], "label": int(lab)}
            for (r,lab) in labeled],
    },
    "detectors": {
        "name_id_fires_on_candidates": len(name_id_fires),
        "n_candidate_pairs": len(cand_rows),
        "model_code_multi_entity_clusters": len(code_clusters),
        "model_code_distinct_name_false_surface": len(false_surface),
        "model_code_false_surface_examples": false_surface[:25],
    },
    "clauses": {
        "lifecycle_transfers": bool(clause1_pass),
        "calibration_breaks": bool(not calibration_transfers),
    },
    "verdict": verdict,
}
out = Path(f"../reports/corpus-transfer-h157-{stamp}.json")
out.write_text(json.dumps(report, indent=1, default=str))
rp(f"report written: [bold]{out}[/bold]")
driver.close()